# Hierarchical Time-Series Forecasting on Walmart M5 Sales

This notebook is a compact, runnable version of the project pipeline. It trains global sequence models on a sampled subset of item-store daily M5 sales, compares them to a seasonal naive baseline, and writes honest result artifacts. The default settings are intentionally small so the notebook can be run on CPU; increase the config values for heavier experiments.


In [ ]:
import math
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

torch = None
nn = None
DataLoader = None
TensorDataset = None
device = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def ensure_torch():
    """Import PyTorch only when the neural-model cells are actually run."""
    global torch, nn, DataLoader, TensorDataset, device
    if torch is None:
        try:
            import torch as _torch
            import torch.nn as _nn
            from torch.utils.data import DataLoader as _DataLoader, TensorDataset as _TensorDataset
        except ImportError as exc:
            raise ImportError(
                "PyTorch is required for the neural models. Install torch before running training cells."
            ) from exc
        torch = _torch
        nn = _nn
        DataLoader = _DataLoader
        TensorDataset = _TensorDataset
        torch.manual_seed(SEED)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
    return torch, nn, DataLoader, TensorDataset, device


In [ ]:
# -------------------------------------------------------------------------
# Config: defaults are intentionally small and CPU-friendly.
# -------------------------------------------------------------------------
M5_DATA_DIR = Path('data/m5')
M5_SALES_EVALUATION_PATH = M5_DATA_DIR / 'sales_train_evaluation.csv'
M5_SALES_VALIDATION_PATH = M5_DATA_DIR / 'sales_train_validation.csv'
M5_SALES_PATH = M5_SALES_EVALUATION_PATH if M5_SALES_EVALUATION_PATH.exists() else M5_SALES_VALIDATION_PATH
M5_CALENDAR_PATH = M5_DATA_DIR / 'calendar.csv'
M5_PRICES_PATH = M5_DATA_DIR / 'sell_prices.csv'

OUTPUT_DIR = Path('outputs/m5')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAST_RUN = True
N_TIME_SERIES = 500 if FAST_RUN else 3000
N_TIMESTEPS = 1000 if FAST_RUN else 1941
SEQ_LEN = 30
HORIZON = 1
TRAIN_FRAC = 0.8
BATCH_SIZE = 512
EPOCHS = 5 if FAST_RUN else 30
LEARNING_RATE = 1e-3
MIN_SERIES_STD = 1.0
SEASONAL_LAG = 7
N_CAL = 4
EMBED_DIM = 4

print({
    'FAST_RUN': FAST_RUN,
    'N_TIME_SERIES': N_TIME_SERIES,
    'N_TIMESTEPS': N_TIMESTEPS,
    'EPOCHS': EPOCHS,
    'OUTPUT_DIR': str(OUTPUT_DIR),
})


In [ ]:
def require_file(path: Path, description: str) -> Path:
    """Fail early with a useful message instead of a cryptic pandas error."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Place the M5 files under data/m5/. Expected at least "
            "sales_train_evaluation.csv; calendar.csv and sell_prices.csv are optional for this notebook."
        )
    return path


def mean_absolute_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean((y_true - y_pred) ** 2))


def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_true - y_pred) / denom) * 100.0)


def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


def metric_row(model_name, y_true, y_pred):
    return {
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'sMAPE (%)': smape(y_true, y_pred),
    }


In [ ]:
def rolling_window_cv(data, window_size=30, overlap=7, horizon=7):
    """
    Rolling window time-series cross-validation.

    Parameters:
    - data: pandas DataFrame or Series sorted by time
    - window_size: training window length
    - overlap: overlap between consecutive windows
    - horizon: forecast length / test period

    Returns:
    - list of (train_idx, test_idx) numpy arrays
    """
    if window_size <= 0 or horizon <= 0:
        raise ValueError('window_size and horizon must be positive.')
    step = window_size - overlap
    if step <= 0:
        raise ValueError('overlap must be smaller than window_size.')

    n = len(data)
    splits = []
    start = 0
    while start + window_size + horizon <= n:
        train_start = start
        train_end = start + window_size
        test_start = train_end
        test_end = train_end + horizon
        train_idx = np.arange(train_start, train_end)
        test_idx = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))
        start += step
    return splits


## Data Loading


In [ ]:
require_file(M5_SALES_PATH, 'M5 sales table')
df_raw = pd.read_csv(M5_SALES_PATH)
print(df_raw.head())
print(df_raw.shape)


In [ ]:
ts_columns = [col for col in df_raw.columns if col.startswith('d_')]
if len(ts_columns) < SEQ_LEN + HORIZON + SEASONAL_LAG:
    raise ValueError(f"Not enough time columns for SEQ_LEN={SEQ_LEN}, HORIZON={HORIZON}.")

ts_columns = ts_columns[-min(N_TIMESTEPS, len(ts_columns)):]
meta_columns = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
missing_meta = [c for c in meta_columns if c not in df_raw.columns]
if missing_meta:
    raise ValueError(f"M5 sales table is missing metadata columns: {missing_meta}")

df = (
    df_raw[meta_columns + ts_columns]
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
    .iloc[:N_TIME_SERIES]
    .copy()
)
print(f"Using {len(df)} series and {len(ts_columns)} time steps.")


In [ ]:
def compute_calendar_features(ts_columns, calendar_path=M5_CALENDAR_PATH):
    """Return deterministic known-future calendar channels for selected d_* columns."""
    if calendar_path.exists():
        cal = pd.read_csv(calendar_path, usecols=['d', 'date'])
        cal['date'] = pd.to_datetime(cal['date'])
        cal = cal.set_index('d').reindex(ts_columns)
        if cal['date'].isna().any():
            missing = cal[cal['date'].isna()].index[:5].tolist()
            raise ValueError(f"calendar.csv does not contain expected day keys, e.g. {missing}")
        dow = cal['date'].dt.dayofweek.to_numpy()
        doy = cal['date'].dt.dayofyear.to_numpy() - 1
    else:
        print('calendar.csv not found; using synthetic day-of-week/day-of-year features.')
        t = np.arange(len(ts_columns))
        dow = t % 7
        doy = t % 365

    return np.stack([
        np.sin(2 * np.pi * dow / 7),
        np.cos(2 * np.pi * dow / 7),
        np.sin(2 * np.pi * doy / 365),
        np.cos(2 * np.pi * doy / 365),
    ], axis=1).astype(np.float32)

calendar_features = compute_calendar_features(ts_columns)
print(calendar_features.shape)


In [ ]:
CATEGORIES = sorted(df['cat_id'].unique())
CAT_TO_ID = {cat: i for i, cat in enumerate(CATEGORIES)}
product_categories = df.set_index('id')['cat_id'].to_dict()
product_stores = df.set_index('id')['store_id'].to_dict()
series_dict = {
    row['id']: row[ts_columns].to_numpy(dtype=np.float32)
    for _, row in df.iterrows()
}
print('Categories:', CAT_TO_ID)


## Rolling Window Baseline CV

This rolling-window check evaluates the cheap seasonal naive baseline across overlapping time windows. It is intentionally not used to retrain LSTM/Transformer models on every fold.


In [ ]:
def rolling_baseline_cv(series_dict, window_size=90, overlap=14, horizon=7,
                        seasonal_lag=SEASONAL_LAG, max_series=100):
    """Lightweight rolling CV for the seasonal naive baseline only."""
    rows = []
    for pid, series in list(series_dict.items())[:max_series]:
        series = np.asarray(series, dtype=float)
        frame = pd.DataFrame({'sales': series})
        for fold, (_, test_idx) in enumerate(rolling_window_cv(frame, window_size, overlap, horizon)):
            pred_idx = test_idx - seasonal_lag
            if pred_idx.min() < 0:
                continue
            actual = series[test_idx]
            pred = series[pred_idx]
            rows.append({
                'id': pid,
                'cat_id': product_categories[pid],
                'store_id': product_stores[pid],
                'fold': fold,
                'window_size': window_size,
                'horizon': horizon,
                'MAE': mean_absolute_error(actual, pred),
                'RMSE': rmse(actual, pred),
                'sMAPE (%)': smape(actual, pred),
            })
    return pd.DataFrame(rows)

rolling_cv_results = rolling_baseline_cv(series_dict)
rolling_cv_results.to_csv(OUTPUT_DIR / 'm5_rolling_baseline_cv.csv', index=False)
rolling_cv_results.head()


## Window Dataset


In [ ]:
def robust_train_scaler(series, split, min_std=MIN_SERIES_STD):
    train_part = np.asarray(series[:split], dtype=np.float32)
    mu = float(np.nanmean(train_part))
    sd = float(np.nanstd(train_part))
    if not np.isfinite(sd) or sd < min_std:
        sd = float(min_std)
    return mu, sd


def build_global_dataset(series_dict, product_categories, seq_len, horizon, train_frac):
    X_tr, y_tr, cat_tr, pid_tr = [], [], [], []
    X_te, y_te, cat_te, pid_te = [], [], [], []
    baseline_te, baseline_pid = [], []
    scalers = {}

    for pid, series in series_dict.items():
        series = np.asarray(series, dtype=np.float32)
        n = len(series)
        split = int(n * train_frac)
        if split <= seq_len + horizon or n - split < horizon:
            continue

        mu, sd = robust_train_scaler(series, split)
        scalers[pid] = (mu, sd)
        scaled_sales = (series - mu) / sd
        full = np.concatenate([scaled_sales[:, None], calendar_features[:n]], axis=1).astype(np.float32)
        cat_id = CAT_TO_ID[product_categories[pid]]

        for i in range(split - seq_len - horizon + 1):
            X_tr.append(full[i:i + seq_len])
            y_tr.append(scaled_sales[i + seq_len:i + seq_len + horizon])
            cat_tr.append(cat_id)
            pid_tr.append(pid)

        for i in range(max(0, split - seq_len), n - seq_len - horizon + 1):
            target_start = i + seq_len
            if target_start < split:
                continue
            X_te.append(full[i:i + seq_len])
            y_te.append(scaled_sales[target_start:target_start + horizon])
            cat_te.append(cat_id)
            pid_te.append(pid)
            if target_start - SEASONAL_LAG >= 0:
                baseline_te.append(series[target_start - SEASONAL_LAG:target_start - SEASONAL_LAG + horizon])
            else:
                baseline_te.append(np.repeat(series[:target_start].mean(), horizon))
            baseline_pid.append(pid)

    return (
        np.asarray(X_tr, dtype=np.float32),
        np.asarray(y_tr, dtype=np.float32),
        np.asarray(cat_tr, dtype=np.int64),
        np.asarray(pid_tr),
        np.asarray(X_te, dtype=np.float32),
        np.asarray(y_te, dtype=np.float32),
        np.asarray(cat_te, dtype=np.int64),
        np.asarray(pid_te),
        np.asarray(baseline_te, dtype=np.float32),
        np.asarray(baseline_pid),
        scalers,
    )



def make_loader(X, y, cat, batch=BATCH_SIZE, shuffle=True):
    ensure_torch()
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(cat, dtype=torch.long),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)


In [ ]:
(
    X_tr, y_tr, cat_tr, pid_tr,
    X_te, y_te, cat_te, pid_te,
    baseline_te, baseline_pid,
    scalers,
) = build_global_dataset(series_dict, product_categories, SEQ_LEN, HORIZON, TRAIN_FRAC)

if len(X_tr) == 0 or len(X_te) == 0:
    raise ValueError('No train/test windows were created. Reduce SEQ_LEN or check the dataset.')

print(f"Train: X={X_tr.shape}, y={y_tr.shape}")
print(f"Test:  X={X_te.shape}, y={y_te.shape}")
tr_loader = make_loader(X_tr, y_tr, cat_tr, shuffle=True)
te_loader = make_loader(X_te, y_te, cat_te, shuffle=False)


## Models


In [ ]:
ensure_torch()

N_TIME_FEATS = 1 + N_CAL

class GlobalLSTM(nn.Module):
    def __init__(self, n_categories, embed_dim=EMBED_DIM, n_time_feats=N_TIME_FEATS,
                 hidden=64, n_layers=2, horizon=1, dropout=0.2):
        super().__init__()
        self.cat_embed = nn.Embedding(n_categories, embed_dim)
        self.lstm = nn.LSTM(
            input_size=n_time_feats + embed_dim,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x_time, cat_id):
        B, T, _ = x_time.shape
        cat_vec = self.cat_embed(cat_id).unsqueeze(1).expand(B, T, -1)
        x = torch.cat([x_time, cat_vec], dim=-1)
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class GlobalTransformer(nn.Module):
    def __init__(self, n_categories, embed_dim=EMBED_DIM, n_time_feats=N_TIME_FEATS,
                 d_model=64, nhead=4, n_layers=2, dim_ff=128, horizon=1, dropout=0.1):
        super().__init__()
        self.cat_embed = nn.Embedding(n_categories, embed_dim)
        self.input_proj = nn.Linear(n_time_feats + embed_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x_time, cat_id):
        B, T, _ = x_time.shape
        cat_vec = self.cat_embed(cat_id).unsqueeze(1).expand(B, T, -1)
        x = torch.cat([x_time, cat_vec], dim=-1)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        return self.head(x[:, -1])


## Training And Evaluation


In [ ]:
def train(model, tr_loader, te_loader, epochs=EPOCHS, lr=LEARNING_RATE, name='model'):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    crit = nn.MSELoss()
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, cat, yb in tr_loader:
            xb, cat, yb = xb.to(device), cat.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb, cat), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(tr_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, cat, yb in te_loader:
                xb, cat, yb = xb.to(device), cat.to(device), yb.to(device)
                val_loss += crit(model(xb, cat), yb).item() * xb.size(0)
        val_loss /= len(te_loader.dataset)
        sched.step()
        history.append({'epoch': ep, 'train_scaled_mse': train_loss, 'val_scaled_mse': val_loss})
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{name}] epoch {ep:3d} | train scaled MSE {train_loss:.4f} | val scaled MSE {val_loss:.4f}")

    return pd.DataFrame(history)


def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, cat, yb in loader:
            preds.append(model(xb.to(device), cat.to(device)).cpu().numpy())
            trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def inverse_scale(values_scaled, pids, scalers):
    values = []
    for value, pid in zip(values_scaled, pids):
        mu, sd = scalers[pid]
        values.append(value * sd + mu)
    return np.asarray(values, dtype=float).ravel()


def evaluate_neural(model, loader, pids, scalers, name):
    preds_s, trues_s = predict_all(model, loader)
    preds = inverse_scale(preds_s, pids, scalers)
    trues = inverse_scale(trues_s, pids, scalers)
    return preds, trues, metric_row(name, trues, preds)


def seasonal_naive_predictions():
    return np.asarray(baseline_te, dtype=float).ravel()


def baseline_metrics():
    trues = inverse_scale(y_te, pid_te, scalers)
    preds = seasonal_naive_predictions()
    return preds, trues, metric_row(f'Baseline lag_{SEASONAL_LAG}', trues, preds)


In [ ]:
baseline_pred, baseline_true, baseline_row = baseline_metrics()
print(pd.DataFrame([baseline_row]))


In [ ]:
print('=' * 70)
print('Training GLOBAL LSTM: category + calendar')
lstm = GlobalLSTM(n_categories=len(CATEGORIES), horizon=HORIZON)
lstm_history = train(lstm, tr_loader, te_loader, name='LSTM')

print('=' * 70)
print('Training GLOBAL Transformer: category + calendar')
tfm = GlobalTransformer(n_categories=len(CATEGORIES), horizon=HORIZON)
tfm_history = train(tfm, tr_loader, te_loader, name='Transformer')


In [ ]:
lstm_pred, y_true, lstm_row = evaluate_neural(lstm, te_loader, pid_te, scalers, 'LSTM')
tfm_pred, _, tfm_row = evaluate_neural(tfm, te_loader, pid_te, scalers, 'Transformer')

result_rows = [baseline_row, lstm_row, tfm_row]
model_results = pd.DataFrame(result_rows).sort_values('RMSE').reset_index(drop=True)
model_results.to_csv(OUTPUT_DIR / 'm5_model_results.csv', index=False)
model_results


## Optional Lightweight LightGBM Baseline

This cell only runs when `lightgbm`, `calendar.csv`, and `sell_prices.csv` are available. It is intentionally small and feature-limited; the older project notebook contains the heavier tabular pipeline.


In [ ]:
def maybe_train_lightgbm():
    if lgb is None:
        print('Skipping LightGBM: package is not installed.')
        return None
    if not (M5_CALENDAR_PATH.exists() and M5_PRICES_PATH.exists()):
        print('Skipping LightGBM: calendar.csv and sell_prices.csv are needed for tabular features.')
        return None

    long_df = df.melt(
        id_vars=meta_columns,
        value_vars=ts_columns,
        var_name='d',
        value_name='sales',
    )
    cal = pd.read_csv(M5_CALENDAR_PATH, usecols=['d', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI'])
    cal['date'] = pd.to_datetime(cal['date'])
    prices = pd.read_csv(M5_PRICES_PATH)
    data = long_df.merge(cal, on='d', how='left').merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
    data = data.sort_values(['id', 'date'])
    data['sell_price'] = data.groupby('id')['sell_price'].ffill().fillna(0.0)
    for lag in [7, 28]:
        data[f'lag_{lag}'] = data.groupby('id')['sales'].shift(lag)
    data['lag_1'] = data.groupby('id')['sales'].shift(1)
    data['roll_mean_7'] = data.groupby('id')['lag_1'].rolling(7).mean().reset_index(level=0, drop=True)
    data = data.dropna(subset=['lag_7', 'lag_28', 'roll_mean_7']).copy()
    split_date = data['date'].quantile(TRAIN_FRAC)
    train_df = data[data['date'] < split_date]
    val_df = data[data['date'] >= split_date]
    features = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday', 'month', 'year', 'sell_price', 'lag_7', 'lag_28', 'roll_mean_7', 'snap_CA', 'snap_TX', 'snap_WI']
    cat_features = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
    for c in cat_features:
        train_df.loc[:, c] = train_df[c].astype('category')
        val_df.loc[:, c] = val_df[c].astype('category')
        val_df.loc[:, c] = val_df[c].cat.set_categories(train_df[c].cat.categories)
    params = {'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.05, 'num_leaves': 31, 'seed': SEED, 'verbosity': -1}
    train_set = lgb.Dataset(train_df[features], train_df['sales'].astype('float32'), categorical_feature=cat_features, free_raw_data=False)
    val_set = lgb.Dataset(val_df[features], val_df['sales'].astype('float32'), categorical_feature=cat_features, free_raw_data=False)
    model = lgb.train(params, train_set, valid_sets=[val_set], num_boost_round=300, callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)])
    pred = model.predict(val_df[features], num_iteration=model.best_iteration)
    return model, metric_row('LightGBM small', val_df['sales'].to_numpy(), pred)

lgb_result = maybe_train_lightgbm()
if lgb_result is not None:
    lgb_model, lgb_row = lgb_result
    model_results = pd.concat([model_results, pd.DataFrame([lgb_row])], ignore_index=True).sort_values('RMSE').reset_index(drop=True)
    model_results.to_csv(OUTPUT_DIR / 'm5_model_results.csv', index=False)
model_results


## Diagnostics And Saved Artifacts


In [ ]:
eval_df = pd.DataFrame({
    'id': pid_te,
    'store_id': [product_stores[pid] for pid in pid_te],
    'cat_id': [product_categories[pid] for pid in pid_te],
    'actual': y_true,
    'lstm_pred': lstm_pred,
    'transformer_pred': tfm_pred,
    'baseline_pred': baseline_pred,
})

category_summary = eval_df.groupby('cat_id').apply(
    lambda g: pd.Series({
        'Transformer_MAE': mean_absolute_error(g['actual'], g['transformer_pred']),
        'Baseline_MAE': mean_absolute_error(g['actual'], g['baseline_pred']),
        'n': len(g),
    })
).reset_index()
store_summary = eval_df.groupby('store_id').apply(
    lambda g: pd.Series({
        'Transformer_MAE': mean_absolute_error(g['actual'], g['transformer_pred']),
        'Baseline_MAE': mean_absolute_error(g['actual'], g['baseline_pred']),
        'n': len(g),
    })
).reset_index()

category_summary.to_csv(OUTPUT_DIR / 'm5_category_summary.csv', index=False)
store_summary.to_csv(OUTPUT_DIR / 'm5_store_summary.csv', index=False)
category_summary


In [ ]:
import matplotlib.pyplot as plt


def forecast_vs_actual(example_id=None):
    if example_id is None:
        example_id = eval_df['id'].iloc[0]
    one = eval_df[eval_df['id'] == example_id].reset_index(drop=True)
    if one.empty:
        raise ValueError(f'No evaluation rows for {example_id}')
    plt.figure(figsize=(10, 4))
    plt.plot(one.index, one['actual'], label='Actual', marker='o')
    plt.plot(one.index, one['transformer_pred'], label='Transformer', marker='o')
    plt.plot(one.index, one['baseline_pred'], label=f'Baseline lag_{SEASONAL_LAG}', marker='o')
    plt.title(f'M5 forecast vs actual: {example_id}')
    plt.xlabel('Evaluation window')
    plt.ylabel('Daily sales')
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / 'm5_forecast_vs_actual.png'
    plt.savefig(path, dpi=150)
    plt.show()
    return path

forecast_plot_path = forecast_vs_actual()
print(forecast_plot_path)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(model_results['Model'], model_results['RMSE'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('RMSE')
plt.title('M5 model comparison')
plt.tight_layout()
model_comparison_path = OUTPUT_DIR / 'm5_model_comparison.png'
plt.savefig(model_comparison_path, dpi=150)
plt.show()
print(model_comparison_path)


In [ ]:
with open(OUTPUT_DIR / 'global_models.pkl', 'wb') as f:
    pickle.dump({
        'lstm_state_dict': lstm.state_dict(),
        'transformer_state_dict': tfm.state_dict(),
        'scalers': scalers,
        'cat_to_id': CAT_TO_ID,
        'config': {
            'seq_len': SEQ_LEN,
            'horizon': HORIZON,
            'n_time_series': N_TIME_SERIES,
            'n_timesteps': N_TIMESTEPS,
            'fast_run': FAST_RUN,
        },
        'model_results': model_results,
    }, f)

print('Saved:')
print(OUTPUT_DIR / 'm5_model_results.csv')
print(OUTPUT_DIR / 'global_models.pkl')
